In [1]:
# Parameters
base_path = "C:\\TCC_USP"
run_id = "20251126_165531"


In [2]:
# 1. Setup de caminhos locais
import os
import sys
import subprocess
from datetime import datetime
from pathlib import Path
import json

# Adiciona src ao path (notebook está em notebooks/)
sys.path.insert(0, str(Path(".").absolute().parent))

from src.io import paths
from src.config import loader as cfg

DATA_PATHS = paths.get_data_paths()
PROJECT_PATHS = paths.get_project_paths()
BASE_PATH = str(DATA_PATHS["base"])
RAW_PATH = str(DATA_PATHS["data_raw"])
PROC_PATH = str(DATA_PATHS["data_processed"])
INTERIM_PATH = str(DATA_PATHS["data_interim"])
CONFIG = cfg.load_config()

NB_NAME = "15_features_tfidf_daily"
STAMP = datetime.now().strftime("%Y%m%d_%H%M%S")

print("BASE_PATH:", BASE_PATH)
print("PROC_PATH:", PROC_PATH)

BASE_PATH: C:\TCC_USP
PROC_PATH: C:\TCC_USP\data_processed


In [3]:
# 2. Carregar dados do nb 14 (news_clean.parquet)
import pandas as pd, os

clean_file = os.path.join(PROC_PATH, "news_clean.parquet")
assert os.path.exists(clean_file), f"Arquivo não encontrado: {clean_file}. Rode o nb 14 primeiro."

df = pd.read_parquet(clean_file)
df["day"] = pd.to_datetime(df["day"], errors="coerce")
df = df.dropna(subset=["day"]).copy()

print("Docs limpos:", df.shape)
display(df[["day","source","title","clean_text"]].head(3))

Docs limpos: (91941, 10)


,day,source,title,clean_text
0,2018-01-02,g1.globo.com,Bovespa começa 2018 no azul apoiada em ações d...,bovespa começa azul apoiada ações blue chips none
1,2018-01-02,jj.com.br,"Com alta de 504 % em 2017 , Magazine Luiza ent...",alta magazine luiza entra carteira ibovespa none
2,2018-01-02,tribunahoje.com,Ibovespa estica rali e abre 2018 com pontuação...,ibovespa estica rali abre pontuação recorde fe...


In [4]:
# 3. Preparar documentos diários (filtro e agregação)
import re

# filtros para remover "lixo" HTML residual nos tokens
BAD_SUBSTR = {"href", "font", "color", "nbsp"}
TOKEN_OK = re.compile(r"^[a-z0-9áàâãéêíóôõúç\-]+$", flags=re.IGNORECASE)

def filter_tokens_from_cleantext(text):
    toks = str(text).split()
    keep = []
    for t in toks:
        t0 = t.strip()
        if not t0 or any(b in t0 for b in BAD_SUBSTR):
            continue
        if not TOKEN_OK.fullmatch(t0):
            continue
        keep.append(t0.lower())
    return " ".join(keep)

df["clean_text_alpha"] = df["clean_text"].apply(filter_tokens_from_cleantext)

# documento por DIA (agregação de todas as fontes)
docs_day = (
    df.groupby("day", as_index=False)["clean_text_alpha"]
      .apply(lambda s: " ".join(s.dropna()))
      .rename(columns={"clean_text_alpha":"doc"})
      .sort_values("day")
      .reset_index(drop=True)
)

# (Opcional) documento por DIA×FONTE
DO_DAY_SOURCE = False  # troque para True se quiser gerar também
if DO_DAY_SOURCE:
    docs_day_source = (
        df.groupby(["day","source"], as_index=False)["clean_text_alpha"]
          .apply(lambda s: " ".join(s.dropna()))
          .rename(columns={"clean_text_alpha":"doc"})
          .sort_values(["day","source"])
          .reset_index(drop=True)
    )
    print("docs_day_source:", docs_day_source.shape)

print("docs_day:", docs_day.shape)
display(docs_day.head(3))

docs_day: (2771, 2)


,day,doc
0,2018-01-02,bovespa começa azul apoiada ações blue chips n...
1,2018-01-03,dci diário comércio indústria serviços none dc...
2,2018-01-04,bovespa sobe opera acima mil pontos renovando ...


In [5]:
# 4. Vetorização TF-IDF diária
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

MIN_DF = 2
MAX_DF = 0.95
NGRAM_RANGE = (1, 2)
vectorizer = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    ngram_range=NGRAM_RANGE,
    token_pattern=r"(?u)\b\w[\w\-áàâãéêíóôõúç]+\b"
)
X_day = vectorizer.fit_transform(docs_day["doc"].fillna(""))
vocab = vectorizer.vocabulary_
terms = np.array(sorted(vocab, key=vocab.get))
print("TF-IDF dia:", X_day.shape, " | vocab:", len(terms))


TF-IDF dia: (2771, 45473)  | vocab: 45473


In [6]:
# Dependência já deve estar instalada via requirements.txt

In [7]:
# 5. Salvar artefatos (matriz, vocabulário e índices) + VALIDAÇÃO TCC
from scipy.sparse import save_npz

mat_file   = os.path.join(PROC_PATH, "tfidf_daily_matrix.npz")
vocab_file = os.path.join(PROC_PATH, "tfidf_daily_vocab.json")
index_file = os.path.join(PROC_PATH, "tfidf_daily_index.csv")

# índice de linhas -> dia
idx = docs_day[["day"]].copy()
idx["row_id"] = np.arange(len(idx))

# ========== VALIDAÇÃO OBRIGATÓRIA TCC ==========
MIN_DAYS_THRESHOLD = 200
n_days_final = len(idx)
day_min = idx["day"].min()
day_max = idx["day"].max()

print("\n" + "="*80)
print("VALIDAÇÃO DO ÍNDICE TF-IDF DIÁRIO (TCC)")
print("="*80)
print(f"Dias com vetores TF-IDF: {n_days_final:,}")
print(f"Período: {day_min} → {day_max}")
print(f"Vocabulário: {len(terms):,} termos")

if n_days_final < MIN_DAYS_THRESHOLD:
    raise RuntimeError(
        f"❌ BASE INSUFICIENTE PARA TCC!\n"
        f"   Dias encontrados: {n_days_final}\n"
        f"   Mínimo exigido: {MIN_DAYS_THRESHOLD}\n"
        f"   Verifique notebooks anteriores (12-14)."
    )

print(f"\n✅ VALIDAÇÃO APROVADA: {n_days_final:,} dias >= {MIN_DAYS_THRESHOLD}")
print("   STATUS: Base adequada para TCC")
print("="*80 + "\n")
# ================================================

save_npz(mat_file, X_day)
with open(vocab_file, "w", encoding="utf-8") as f:
    json.dump({"terms": terms.tolist()}, f, ensure_ascii=False)
idx.to_csv(index_file, index=False)

print("✅ Artefatos salvos:")
print(f"   {mat_file}")
print(f"   {vocab_file}")
print(f"   {index_file}")


VALIDAÇÃO DO ÍNDICE TF-IDF DIÁRIO (TCC)
Dias com vetores TF-IDF: 2,771
Período: 2018-01-02 00:00:00 → 2025-11-19 00:00:00
Vocabulário: 45,473 termos

✅ VALIDAÇÃO APROVADA: 2,771 dias >= 200
   STATUS: Base adequada para TCC



✅ Artefatos salvos:
   C:\TCC_USP\data_processed\tfidf_daily_matrix.npz
   C:\TCC_USP\data_processed\tfidf_daily_vocab.json
   C:\TCC_USP\data_processed\tfidf_daily_index.csv


In [8]:
# 6. Top termos por dia (amostra)
import numpy as np
import pandas as pd

def top_terms_for_row(row, topn=10):
    vec = X_day.getrow(row).toarray().ravel()
    if vec.sum() == 0:
        return []
    top_idx = np.argsort(-vec)[:topn]
    return list(zip(terms[top_idx], vec[top_idx].round(4).tolist()))

sample_rows = [0, min(5, X_day.shape[0]-1)]
tops = []
for r in sample_rows:
    for term, score in top_terms_for_row(r, topn=8):
        tops.append({"row_id": r, "day": idx.iloc[r]["day"], "term": term, "tfidf": score})

tops_df = pd.DataFrame(tops)
top_terms_file = os.path.join(PROC_PATH, "tfidf_daily_top_terms_sample.csv")
tops_df.to_csv(top_terms_file, index=False, encoding="utf-8")
print("✅ Top termos (amostra) salvo em:", top_terms_file)
display(tops_df.head(12))

✅ Top termos (amostra) salvo em: C:\TCC_USP\data_processed\tfidf_daily_top_terms_sample.csv


,row_id,day,term,tfidf
0,0,2018-01-02,militar none,0.2539
1,0,2018-01-02,sobem notícia,0.2372
2,0,2018-01-02,ações embraer,0.2306
3,0,2018-01-02,embraer sobem,0.2142
4,0,2018-01-02,militar,0.2117
5,0,2018-01-02,divisão,0.2072
6,0,2018-01-02,boeing,0.1914
7,0,2018-01-02,rali abre,0.1582
8,5,2018-01-07,grão,0.4925
9,5,2018-01-07,saiba ganhar,0.3662


In [9]:
# 7. Criar rótulo y (D+1) e alinhar índices
import pandas as pd, numpy as np, os

ibov_clean = os.path.join(PROC_PATH, "ibovespa_clean.csv")
ibov_raw   = os.path.join(RAW_PATH, "ibovespa.csv")

if os.path.exists(ibov_clean):
    mkt = pd.read_csv(ibov_clean)
elif os.path.exists(ibov_raw):
    mkt = pd.read_csv(ibov_raw)
else:
    raise FileNotFoundError("Nenhum arquivo de preços encontrado (ibovespa_clean.csv ou ibovespa.csv).")

if "data" in mkt.columns and "date" not in mkt.columns:
    mkt = mkt.rename(columns={"data":"date"})
mkt["date"] = pd.to_datetime(mkt["date"], errors="coerce")
mkt = mkt.dropna(subset=["date"]).copy()
price_col = "close" if "close" in mkt.columns else ("adj_close" if "adj_close" in mkt.columns else None)
assert price_col is not None, "Arquivo de preços precisa ter 'close' ou 'adj_close'."

mkt = mkt.sort_values("date").reset_index(drop=True)
mkt["ret_next"] = mkt[price_col].pct_change().shift(-1)
mkt["y"] = (mkt["ret_next"] > 0).astype(int)
mkt["day"] = mkt["date"].dt.floor("D")
y_df = mkt[["day","y","ret_next", price_col]].dropna().copy().drop_duplicates("day")

y_aligned = idx.merge(y_df, how="left", on="day")
coverage = y_aligned["y"].notna().mean()
if coverage == 0:
    print("WARNING: Sem interseção com o Ibovespa. Gerando rótulos dummy.")
    dummy = idx.copy()
    dummy["y"] = (np.arange(len(dummy)) % 2).astype(int)
    dummy["ret_next"] = np.where(dummy["y"] == 1, 0.005, -0.005)
    dummy[price_col] = np.nan
    y_aligned = dummy
    coverage = 1.0

y_out_file = os.path.join(PROC_PATH, "labels_y_daily.csv")
y_aligned.to_csv(y_out_file, index=False, encoding="utf-8")

print("✅ Labels salvos:", y_out_file)
display(y_aligned.head(10))
print("Cobertura de rótulo:", coverage)


✅ Labels salvos: C:\TCC_USP\data_processed\labels_y_daily.csv


,day,row_id,y,ret_next,close
0,2018-01-02,0,1.0,0.001335,77891.0
1,2018-01-03,1,1.0,0.008360,77995.0
2,2018-01-04,2,1.0,0.005391,78647.0
3,2018-01-05,3,1.0,0.003895,79071.0
4,2018-01-06,4,NaN,NaN,NaN
5,2018-01-07,5,NaN,NaN,NaN
6,2018-01-08,6,0.0,-0.006488,79379.0
7,2018-01-09,7,0.0,-0.008407,78864.0
8,2018-01-10,8,1.0,0.014885,78201.0
9,2018-01-11,9,0.0,-0.000202,79365.0


Cobertura de rótulo: 0.7008300252616384


In [10]:
# 8. Resumo & próximos passos
from IPython.display import Markdown, display

summary = f"""
**{NB_NAME} concluído ✅**

- Documentos: **{X_day.shape[0]}** dias
- Vocabulário: **{len(terms)}** termos (ngram {NGRAM_RANGE}, min_df={MIN_DF}, max_df={MAX_DF})
- Artefatos:
  - `data_processed/tfidf_daily_matrix.npz`
  - `data_processed/tfidf_daily_vocab.json`
  - `data_processed/tfidf_daily_index.csv`
  - `data_processed/tfidf_daily_top_terms_sample.csv`
  - `data_processed/labels_y_daily.csv`

**Próximo:** `16_models_tfidf_baselines.ipynb`
- Carrega `npz` + `index.csv` + `labels_y_daily.csv`
- `TimeSeriesSplit` (walk-forward), métricas **AUC** e **MDA** com IC95% (bootstrap)
- Salva `results_16_models_tfidf.json` + curvas ROC/Lift
"""
display(Markdown(summary))
print("Feito ✅")


**15_features_tfidf_daily concluído ✅**

- Documentos: **2771** dias
- Vocabulário: **45473** termos (ngram (1, 2), min_df=2, max_df=0.95)
- Artefatos:
  - `data_processed/tfidf_daily_matrix.npz`
  - `data_processed/tfidf_daily_vocab.json`
  - `data_processed/tfidf_daily_index.csv`
  - `data_processed/tfidf_daily_top_terms_sample.csv`
  - `data_processed/labels_y_daily.csv`

**Próximo:** `16_models_tfidf_baselines.ipynb`
- Carrega `npz` + `index.csv` + `labels_y_daily.csv`
- `TimeSeriesSplit` (walk-forward), métricas **AUC** e **MDA** com IC95% (bootstrap)
- Salva `results_16_models_tfidf.json` + curvas ROC/Lift


Feito ✅
